In [2]:
import pandas as pd
import numpy as np

# Load cleaned dataset
data = pd.read_csv("data\cleaned_data.csv")

# Quick look
data.head()

,qty,total_price,freight_price,unit_price,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_score,customers,...,log_qty,log_price,product_category_name_computers_accessories,product_category_name_consoles_games,product_category_name_cool_stuff,product_category_name_furniture_decor,product_category_name_garden_tools,product_category_name_health_beauty,product_category_name_perfumery,product_category_name_watches_gifts
0,-1.148998,45.95,15.100000,-0.93562,39,161,2,350,4.0,57,...,0.693147,3.849083,False,False,False,False,False,False,False,False
1,-0.935667,137.85,12.933333,-0.93562,39,161,2,350,4.0,61,...,1.386294,3.849083,False,False,False,False,False,False,False,False
2,-0.615670,275.70,14.840000,-0.93562,39,161,2,350,4.0,123,...,1.945910,3.849083,False,False,False,False,False,False,False,False
3,-0.829001,183.80,14.287500,-0.93562,39,161,2,350,4.0,90,...,1.609438,3.849083,False,False,False,False,False,False,False,False
4,-1.042333,91.90,15.100000,-0.93562,39,161,2,350,4.0,54,...,1.098612,3.849083,False,False,False,False,False,False,False,False


Create New Features
1) Price per Customer

In [3]:
data['price_per_customer'] = data['total_price'] / (data['customers'] + 1)

Competitor Price Gap

In [4]:
data['comp1_gap'] = data['unit_price'] - data['comp_1']
data['comp2_gap'] = data['unit_price'] - data['comp_2']
data['comp3_gap'] = data['unit_price'] - data['comp_3']

Price Change from Previous Month

In [5]:
data['price_change'] = data['unit_price'] - data['lag_price']

Discount Indicator

In [6]:
data['discount_flag'] = np.where(data['price_change'] < 0, 1, 0)

Seasonal Features

In [7]:
# Already have month and year from Day 53
# Create quarter
data['quarter'] = pd.to_datetime(data['month']).dt.quarter if 'month' not in data else (data['month']-1)//3 +1

Interaction Features
 Price x Freight

In [8]:
data['price_freight_interaction'] = data['unit_price'] * data['freight_price']

Competitor Gap x Discount

In [9]:
data['comp1_discount_interaction'] = data['comp1_gap'] * data['discount_flag']
data['comp2_discount_interaction'] = data['comp2_gap'] * data['discount_flag']
data['comp3_discount_interaction'] = data['comp3_gap'] * data['discount_flag']

Feature Importance Analysis

In [10]:
from sklearn.ensemble import RandomForestRegressor

# Prepare features and target
X = data.drop(['qty', 'log_qty'], axis=1)
y = data['log_qty']  # using log transformed qty from Day 53

# Train RandomForest for feature importance
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Feature importance
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(10))

total_price                  0.786722
price_freight_interaction    0.064517
unit_price                   0.037780
log_price                    0.033650
lag_price                    0.029834
price_change                 0.025090
customers                    0.002064
s                            0.002058
product_name_lenght          0.001359
freight_price                0.001107
dtype: float64


Save Feature Engineered Dataset

In [13]:
data.to_csv("data/feature_engineered_data.csv", index=False)